In [60]:
%pip install -U langchain-groq langchain-chroma langchain-cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 3.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [langchain-cohere][cohere]

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [61]:
import os
from getpass import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter Groq API key: ")

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass("Enter Cohere API key: ")

print("Groq key configured:", bool(os.environ.get("GROQ_API_KEY")))
print("Cohere key configured:", bool(os.environ.get("COHERE_API_KEY")))

Groq key configured: True
Cohere key configured: True


In [62]:
from langchain_groq import ChatGroq
from langchain_cohere import CohereEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

# Hosted chat generation via Groq
chat_model = ChatGroq(model="openai/gpt-oss-120b")

# Hosted embeddings via Cohere API — no local embedding model is downloaded.
embeddings = CohereEmbeddings(
    model="embed-english-v3.0",
    cohere_api_key=os.environ["COHERE_API_KEY"],
)

# Quick test of Groq chat
print(chat_model.invoke("What is the capital of India?").content)

# Quick test of hosted embeddings (prints only vector dimensions)
test_vector = embeddings.embed_query("Test sentence for semantic search")
print("Embedding dimensions:", len(test_vector))

The capital of India is **New Delhi**.
Embedding dimensions: 1024


In [63]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )


In [64]:
docs = [doc1,doc2,doc3,doc4,doc5]

In [65]:
vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="my_chroma_db_groq_embeddings",
    collection_name="ipl_players",
)

In [66]:
vector_store.add_documents(documents=docs)
print("Documents in Chroma:", vector_store._collection.count())

Documents in Chroma: 5


In [67]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['229c550e-59a3-488b-a3c7-7da5295b155b',
  '533924e7-cda2-4d8b-83f9-63a203588a9b',
  'aa9a1251-f14e-4e2a-a252-1a8a1c425422',
  'daa9e553-ba55-4273-85a9-0f5a77b1fce5',
  '6dacc2dd-f27c-450d-89ed-fc0439b3165c'],
 'embeddings': array([[ 0.01044464, -0.00331116, -0.01013947, ...,  0.08074951,
          0.04669189,  0.04876709],
        [ 0.01042175,  0.00094128,  0.01387787, ...,  0.11340332,
          0.05953979,  0.04449463],
        [-0.00927734,  0.01901245, -0.00845337, ...,  0.09735107,
         -0.00563431,  0.01727295],
        [ 0.00107193,  0.01246643, -0.00303268, ...,  0.05105591,
          0.00432587,  0.01430511],
        [-0.01687622,  0.02511597, -0.05059814, ...,  0.12805176,
          0.03086853,  0.03089905]], shape=(5, 1024)),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the m

In [68]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(id='daa9e553-ba55-4273-85a9-0f5a77b1fce5', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(id='6dacc2dd-f27c-450d-89ed-fc0439b3165c', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [71]:
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(id='daa9e553-ba55-4273-85a9-0f5a77b1fce5', metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9489356279373169),
 (Document(id='6dacc2dd-f27c-450d-89ed-fc0439b3165c', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.2517635822296143)]

In [77]:
vector_store.similarity_search(
    query=" ",
    filter={"team": "Chennai Super Kings"},
)

[Document(id='aa9a1251-f14e-4e2a-a252-1a8a1c425422', metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
 Document(id='6dacc2dd-f27c-450d-89ed-fc0439b3165c', metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.')]

In [78]:
updated_doc1 = Document(
    page_content=(
        "Virat Kohli is a former captain of Royal Challengers Bangalore (RCB), renowned for "
        "aggressive leadership and consistent batting. His ability to chase targets and anchor "
        "innings has made him one of the dependable players in T20 cricket."
    ),
    metadata={"team": "Royal Challengers Bangalore", "player": "Virat Kohli"},
)

vector_store.update_document(document_id="229c550e-59a3-488b-a3c7-7da5295b155b", document=updated_doc1)


In [79]:
vector_store.get(include=["embeddings","documents", "metadatas"])

{'ids': ['229c550e-59a3-488b-a3c7-7da5295b155b',
  '533924e7-cda2-4d8b-83f9-63a203588a9b',
  'aa9a1251-f14e-4e2a-a252-1a8a1c425422',
  'daa9e553-ba55-4273-85a9-0f5a77b1fce5',
  '6dacc2dd-f27c-450d-89ed-fc0439b3165c'],
 'embeddings': array([[-0.00770569, -0.02780151,  0.03019714, ...,  0.08551025,
          0.0295105 ,  0.0486145 ],
        [ 0.01042175,  0.00094128,  0.01387787, ...,  0.11340332,
          0.05953979,  0.04449463],
        [-0.00927734,  0.01901245, -0.00845337, ...,  0.09735107,
         -0.00563431,  0.01727295],
        [ 0.00107193,  0.01246643, -0.00303268, ...,  0.05105591,
          0.00432587,  0.01430511],
        [-0.01687622,  0.02511597, -0.05059814, ...,  0.12805176,
          0.03086853,  0.03089905]], shape=(5, 1024)),
 'documents': ['Virat Kohli is a former captain of Royal Challengers Bangalore (RCB), renowned for aggressive leadership and consistent batting. His ability to chase targets and anchor innings has made him one of the dependable players in 

In [82]:
vector_store.delete(ids = ["229c550e-59a3-488b-a3c7-7da5295b155b"])

In [83]:
vector_store.get(include=["embeddings","documents", "metadatas"])

{'ids': ['533924e7-cda2-4d8b-83f9-63a203588a9b',
  'aa9a1251-f14e-4e2a-a252-1a8a1c425422',
  'daa9e553-ba55-4273-85a9-0f5a77b1fce5',
  '6dacc2dd-f27c-450d-89ed-fc0439b3165c'],
 'embeddings': array([[ 0.01042175,  0.00094128,  0.01387787, ...,  0.11340332,
          0.05953979,  0.04449463],
        [-0.00927734,  0.01901245, -0.00845337, ...,  0.09735107,
         -0.00563431,  0.01727295],
        [ 0.00107193,  0.01246643, -0.00303268, ...,  0.05105591,
          0.00432587,  0.01430511],
        [-0.01687622,  0.02511597, -0.05059814, ...,  0.12805176,
          0.03086853,  0.03089905]], shape=(4, 1024)),
 'documents': ["Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
  'MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.',
  'Jasprit Bumrah i